# NEXUS § 131 — Zero routes: BBP is the wheel plus a route to nothing

> *"There are no dead ends just transformations that exist in different directions. You're reading the end of a different transformation route. As it gets further up you get more degrees of freedom… anything can collide with anything in any state. Deformation is passive and computation is active via the interface."* — Dean, 17 Oct

GPT's § 126 / § 128 runs left two links open:

1. the identity binding the $s$-wheel's base-16 regrouping to the four-lane BBP machine;
2. one theorem joining § 117's register stack to the phase-wheel boundary.

This notebook closes the first exactly. It runs the second along the register chain and states its scope.

**Sealed before any run:** `PREDICTION_zero_route.txt`, SHA-256 `e80c1cc4b724db85a759c3568ebb0ae5e39a66352e94b088040900c9d03bd63d`, sealed at 07:13:01 UTC. The seal text is reproduced in the last cell. Every number below is printed by this notebook's own execution. Anything found after the run is marked **unsealed**.

Ledger words: STANDS (established mathematics) / RUNS (executed here) / OPEN.

In [1]:
RUN_SLOW = True   # n = 24, 32 PSLQ (several minutes); set False for a quick run

In [2]:
# ---- library (self-contained: standard library + sympy + mpmath) ----
import math, time, hashlib
from fractions import Fraction as Fr
import sympy as sp
from mpmath import mp, mpf, mpc, pi, log, sqrt, atan, exp, pslq, fabs, nstr, nint, log10

mp.dps = 420
TOL = mpf(10) ** -380
x, r = sp.symbols('x r')
S2, I = sp.sqrt(2), sp.I

def lanes(n, b, dps=None):
    # Lam_j = sum_k b^-k / (n k + j), j = 1..n
    b = mpf(b); terms = int((dps or mp.dps) / float(log10(abs(b)))) + 12
    return [sum(1 / (b ** k * (n * k + j)) for k in range(terms)) for j in range(1, n + 1)]

def num(t):                       # exact rational -> mpf
    t = sp.Rational(t); return mpf(t.p) / mpf(t.q)

def dot(c, L): return sum(num(t) * l for t, l in zip(c, L))

def poly(c):                      # lane vector -> lane polynomial P_c(x) = sum_j c_j x^j
    return sp.expand(sum(sp.Rational(t) * x ** (j + 1) for j, t in enumerate(c)))

def reldim(vals, maxcoeff=10**12, maxsteps=10**7):
    # dimension of the integer relation space, by PSLQ with elimination (|coeff| bounded)
    idx, rels = list(range(len(vals))), []
    while len(idx) >= 2:
        rr = pslq([vals[i] for i in idx], maxcoeff=maxcoeff, maxsteps=maxsteps)
        if rr is None: break
        assert fabs(sum(a * vals[i] for a, i in zip(rr, idx))) < mpf(10) ** (-(mp.dps * 3 // 4))
        rels.append({i: a for i, a in zip(idx, rr) if a})
        idx.remove(max(i for i, a in zip(idx, rr) if a))
    return len(rels), rels

def span_basis(vals, maxcoeff=10**12, maxsteps=10**7):
    # relations plus a Q-basis of the span (indices that survive elimination)
    idx, rels = list(range(len(vals))), []
    while len(idx) >= 2:
        rr = pslq([vals[i] for i in idx], maxcoeff=maxcoeff, maxsteps=maxsteps)
        if rr is None: break
        assert fabs(sum(a * vals[i] for a, i in zip(rr, idx))) < mpf(10) ** (-(mp.dps * 3 // 4))
        rels.append({i: a for i, a in zip(idx, rr) if a})
        idx.remove(max(i for i, a in zip(idx, rr) if a))
    return rels, idx

def in_span(cval, vals, basis, maxcoeff=10**12):
    # one PSLQ call against the surviving basis: does cval lie in the Q-span?
    return pslq([cval] + [vals[i] for i in basis], maxcoeff=maxcoeff, maxsteps=10**7) is not None

def galois_orbits(n):
    # orbits of m under zeta_n^m -> zeta_n^(a m), sqrt2 -> eps_a sqrt2  (beta_m -> +-beta_(a m + [eps=-1] n/2))
    N = math.lcm(n, 8); seen, orbs = set(), []
    for m in range(n):
        if m in seen: continue
        o = {(a * m + (n // 2 if a % 8 in (3, 5) else 0)) % n for a in range(1, N) if math.gcd(a, N) == 1}
        seen |= o; orbs.append(sorted(o))
    return orbs

checks = {}
def check(name, ok):
    checks[name] = bool(ok); print(("PASS " if ok else "FAIL ") + name)

c_BBP = [4, 0, 0, -2, -1, -1, 0, 0]
Z     = [-8, 8, 4, 8, 2, 2, -1, 0]
c_W   = [2, 2, 1, 0, sp.Rational(-1, 2), sp.Rational(-1, 2), sp.Rational(-1, 4), 0]
def c_of(t): return [sp.Rational(b) + sp.Rational(t) * z for b, z in zip(c_BBP, Z)]
L8 = lanes(8, 16)
print("library ready; mp.dps =", mp.dps)

library ready; mp.dps = 420


## 1 · The π-line

The lanes are $\Lambda_j=\sum_{k\ge0}16^{-k}/(8k+j)$ for $j=1,\dots,8$. A machine is a lane vector $c$, and its output is $c\cdot\Lambda$.

* BBP: $c_{\rm BBP}=(4,0,0,-2,-1,-1,0,0)$.
* The $s$-wheel's own base-16 regrouping. This is GPT's $A_{2r}-\tfrac14A_{2r+1}$, times 4: $c_W=(2,2,1,0,-\tfrac12,-\tfrac12,-\tfrac14,0)$.

Their difference is BBP's own zero relation $Z=(-8,8,4,8,2,2,-1,0)$ (Bailey–Borwein–Plouffe 1997, STANDS):

$$c_W=c_{\rm BBP}+\tfrac14 Z,\qquad Z\cdot\Lambda=0 .$$

$Z$ is not an absence. It is a complete lane transformation whose endpoint is $0$. Every $c(r)=c_{\rm BBP}+rZ$ computes π, so the π machines form a line, and all of its points collide at π.

In [3]:
check("P1 c_W - c_BBP = Z/4 (exact)", all(sp.Rational(w) - b == sp.Rational(z, 4) for w, b, z in zip(c_W, c_BBP, Z)))
for name, cv in [("BBP  r=0  ", c_BBP), ("wheel r=1/4", c_W), ("mirror r=1/2", c_of(sp.Rational(1, 2)))]:
    e = fabs(dot(cv, L8) - pi); print(f"   {name}: |c.Lam - pi| = {nstr(e, 3)}"); check(f"P1 {name.strip()} computes pi", e < TOL)
e = fabs(dot(Z, L8)); print(f"   Z: |Z.Lam| = {nstr(e, 3)}"); check("P1 Z.Lam = 0", e < TOL)

PASS P1 c_W - c_BBP = Z/4 (exact)
   BBP  r=0  : |c.Lam - pi| = 5.78e-421
PASS P1 BBP  r=0 computes pi
   wheel r=1/4: |c.Lam - pi| = 8.67e-421
PASS P1 wheel r=1/4 computes pi
   mirror r=1/2: |c.Lam - pi| = 2.31e-420
PASS P1 mirror r=1/2 computes pi
   Z: |Z.Lam| = 5.51e-420
PASS P1 Z.Lam = 0


## 2 · A machine is a polynomial; the eight lanes are eight wheels

Take any $w$ with $w^8=1/16$. Then $\sum_{r\ge1}w^r/r=\sum_{j=1}^{8}w^j\Lambda_j$. There are eight such wheels, $w_m=\zeta_8^m/\sqrt2$. Inverting the 8-point DFT gives

$$\boxed{\;c\cdot\Lambda=\frac18\sum_{m=0}^{7}P_c(\gamma_m)\,\bigl(-\log(1-w_m)\bigr),\qquad P_c(x)=\sum_j c_jx^j,\quad \gamma_m=\frac1{w_m}=\sqrt2\,\zeta_8^{-m}.\;}$$

The $\gamma_m$ are the roots of

$$x^8-16=(x^2-2)(x^2+2)(x^2-2x+2)(x^2+2x+2),$$

which gives four Galois orbits. A machine is read orbit by orbit, through its polynomial evaluated at the orbit's roots. Each wheel returns through $1-w_m=\beta_m/\sqrt2$, with $\beta_m=\sqrt2-\zeta_8^m$.

The wheel $w_1=(1+i)/2$ is § 125's $s$.

In [4]:
z8 = exp(2j * pi / 8)
w = [z8 ** m / sqrt(2) for m in range(8)]
gam = [1 / wm for wm in w]
Lm = [-log(1 - wm) for wm in w]
def orbit_read(c):
    P = poly(c); f = sp.lambdify(x, P, "mpmath")
    return sum(f(g) * l for g, l in zip(gam, Lm)) / 8
for name, cv in [("BBP", c_BBP), ("wheel", c_W), ("Z", Z)]:
    e = fabs(orbit_read(cv) - dot(cv, L8)); check(f"orbit reading = lane reading for {name} (|diff| {nstr(e,2)})", e < TOL)
fl = sp.factor_list(x**8 - 16)[1]
print("x^8 - 16 =", " * ".join(f"({g})" for g, _ in fl))
check("P3 x^8-16 splits into four quadratic orbits", sorted(sp.degree(g, x) for g, _ in fl) == [2, 2, 2, 2])
print("s = w_1 =", nstr(w[1], 12), "  (1+i)/2 :", fabs(w[1] - mpc(0.5, 0.5)) < TOL)

PASS orbit reading = lane reading for BBP (|diff| 2.7e-420)
PASS orbit reading = lane reading for wheel (|diff| 2.0e-420)
PASS orbit reading = lane reading for Z (|diff| 4.2e-420)
x^8 - 16 = (x**2 - 2*x + 2) * (x**2 - 2) * (x**2 + 2) * (x**2 + 2*x + 2)
PASS P3 x^8-16 splits into four quadratic orbits
s = w_1 = (0.5 + 0.5j)   (1+i)/2 : True


## 3 · The four returns, and where the collision lives

| orbit (roots $\gamma$) | $m$ | return $\beta_m=\sqrt2-\zeta_8^m$ | norm in $\mathbb Q(\zeta_8)$ | what it carries |
|---|---|---|---|---|
| $\pm\sqrt2$ | 0, 4 | $\sqrt2\mp1$ — **unit**: the scale wheel | 1 | $\ln2$, $\sqrt2\ln(1+\sqrt2)$ |
| $1\pm i$ | 1, 7 | $\zeta_8^{\mp1}$ — **torsion**: the phase wheel $s,\bar s$ | 1 | $\ln2$, $\pi$ |
| $\pm\sqrt2\,i$ | 2, 6 | $\sqrt2\mp i$ — **prime 3** | 9 | $\ln3-\ln2$, $\sqrt2\arctan(1/\sqrt2)$ |
| $-1\pm i$ | 3, 5 | $\zeta_8^{\mp1}(2\pm i)$ — **prime 5** | 25 | $\ln5-\ln2$, $\arctan(1/3)$ |

$$\prod_m\beta_m=15=2^4-1 .$$

That is 8 lanes and 7 constants: $\{\ln2,\ \pi,\ \sqrt2\ln(1+\sqrt2),\ \ln3,\ \sqrt2\arctan\tfrac1{\sqrt2},\ \ln5,\ \arctan\tfrac13\}$. The seven are independent over $\overline{\mathbb Q}$ by Baker's theorem (STANDS): they are logarithms of multiplicatively independent algebraic numbers, with $i\pi=\log(-1)$. So the zero-route space is exactly $8-7=1$ dimensional.

**The origin of $Z$.** Two returns cost no prime: the phase wheel (torsion) and the scale wheel (unit $1+\sqrt2$). Both carry the same bit, $\ln2$ (the $\sqrt2$ in $1-w=\beta/\sqrt2$). $Z$ is the route on which the phase wheel's $\ln2$ and the scale wheel's $\ln2$ annihilate while both prime returns stay silent. The prime returns each own a private address, so they cannot collide. The two prime-free returns can, and they do.

In [5]:
z8s = sp.exp(I * sp.pi / 4)
print(" m | beta_m                    | minimal polynomial        | norm Q(z8)")
norms = {}
for m in range(8):
    b = sp.nsimplify(sp.expand_complex(S2 - z8s ** m)); mpol = sp.minimal_polynomial(b, x)
    cf = sp.Poly(mpol, x).all_coeffs(); d = len(cf) - 1
    norms[m] = ((-1) ** d * cf[-1] / cf[0]) ** (4 // d)
    print(f" {m} | {str(sp.simplify(b)):25s} | {str(mpol):25s} | {norms[m]}")
check("P3 returns: {0,4} units, {1,7} torsion, {2,6} norm 9, {3,5} norm 25",
      [abs(norms[m]) for m in range(8)] == [1, 1, 9, 25, 1, 25, 9, 1])
check("P3 prod beta_m = 15 = 2^4 - 1", sp.simplify(sp.expand(sp.prod([S2 - z8s ** m for m in range(8)])) - 15) == 0)

# exact 8 x 7 matrix: Lam_j = sum_t N[j][t] kappa_t, from the orbit reading of P = x^j
kap_names = ["ln2", "pi", "sqrt2*ln(1+sqrt2)", "ln3", "sqrt2*atan(1/sqrt2)", "ln5", "atan(1/3)"]
kap = [log(2), pi, sqrt(2) * log(1 + sqrt(2)), log(3), sqrt(2) * atan(1 / sqrt(2)), log(5), atan(mpf(1) / 3)]
Nrows = []
for j in range(1, 9):
    a = sp.expand(S2 ** j); A, B = a.coeff(S2, 0), a.coeff(S2, 1)
    g = sp.expand((1 - I) ** j); u, v = sp.re(g), sp.im(g)
    h = sp.expand((S2 * I) ** j); A2, B2 = sp.re(h), sp.simplify(sp.im(h) / S2)
    k = sp.expand((-1 - I) ** j); U, V = sp.re(k), sp.im(k)
    Nrows.append([sp.nsimplify(t) for t in [(A + u + A2 + U) / 8, -v / 16, B / 4, -A2 / 8, B2 / 4, -U / 8, -V / 4]])
N = sp.Matrix(Nrows)
for j in range(8):
    print(f"   Lam_{j+1} = " + " + ".join(f"({t}){nm}" for t, nm in zip(Nrows[j], kap_names) if t != 0))
check("P7 each Lam_j is the exact rational combination above (420 digits)",
      all(fabs(sum(num(t) * kk for t, kk in zip(Nrows[j], kap)) - L8[j]) < TOL for j in range(8)))
ker = N.T.nullspace()
check("P7 rank N = 7; exact left kernel = span(Z)", N.rank() == 7 and len(ker) == 1 and
      list(ker[0] * (sp.Integer(Z[0]) / ker[0][0])) == [sp.Integer(t) for t in Z])
e_pi = [0, 1, 0, 0, 0, 0, 0]
check("P7 BBP and wheel both solve c^T N = e_pi",
      list(sp.Matrix([c_BBP]) * N) == e_pi and list(sp.Matrix([c_W]) * N) == e_pi)
d0, r0 = reldim(kap); print("   PSLQ on the 7 constants:", d0, r0)
check("P7 no integer relation among the 7 constants (|coeff| <= 1e12, 420 digits)", d0 == 0)
d1, r1 = reldim(L8); print("   PSLQ on Lam_1..8:", d1, r1)
check("P7 dim rel(Lam) = 1 (it is Z)", d1 == 1)
d2, r2 = reldim([pi] + L8); print("   PSLQ on (pi, Lam_1..8):", d2, r2)
check("P7 dim rel(pi, Lam) = 2", d2 == 2)

 m | beta_m                    | minimal polynomial        | norm Q(z8)


 0 | -1 + sqrt(2)              | x**2 + 2*x - 1            | 1


 1 | sqrt(2)*(1 - I)/2         | x**4 + 1                  | 1
 2 | sqrt(2) - I               | x**4 - 2*x**2 + 9         | 9
 3 | sqrt(2)*(3 - I)/2         | x**4 - 8*x**2 + 25        | 25


 4 | 1 + sqrt(2)               | x**2 - 2*x - 1            | 1
 5 | sqrt(2)*(3 + I)/2         | x**4 - 8*x**2 + 25        | 25
 6 | sqrt(2) + I               | x**4 - 2*x**2 + 9         | 9


 7 | sqrt(2)*(1 + I)/2         | x**4 + 1                  | 1
PASS P3 returns: {0,4} units, {1,7} torsion, {2,6} norm 9, {3,5} norm 25
PASS P3 prod beta_m = 15 = 2^4 - 1
   Lam_1 = (1/16)pi + (1/4)sqrt2*ln(1+sqrt2) + (1/4)sqrt2*atan(1/sqrt2) + (1/8)ln5 + (1/4)atan(1/3)
   Lam_2 = (1/8)pi + (1/4)ln3 + (-1/2)atan(1/3)
   Lam_3 = (1/8)pi + (1/2)sqrt2*ln(1+sqrt2) + (-1/2)sqrt2*atan(1/sqrt2) + (-1/4)ln5 + (1/2)atan(1/3)
   Lam_4 = (-1/2)ln3 + (1/2)ln5
   Lam_5 = (-1/4)pi + (1)sqrt2*ln(1+sqrt2) + (1)sqrt2*atan(1/sqrt2) + (-1/2)ln5 + (-1)atan(1/3)
   Lam_6 = (-1/2)pi + (1)ln3 + (2)atan(1/3)
   Lam_7 = (-1/2)pi + (2)sqrt2*ln(1+sqrt2) + (-2)sqrt2*atan(1/sqrt2) + (1)ln5 + (-2)atan(1/3)
   Lam_8 = (8)ln2 + (-2)ln3 + (-2)ln5
PASS P7 each Lam_j is the exact rational combination above (420 digits)
PASS P7 rank N = 7; exact left kernel = span(Z)
PASS P7 BBP and wheel both solve c^T N = e_pi


   PSLQ on the 7 constants: 0 []
PASS P7 no integer relation among the 7 constants (|coeff| <= 1e12, 420 digits)


   PSLQ on Lam_1..8: 1 [{0: -8, 1: 8, 2: 4, 3: 8, 4: 2, 5: 2, 6: -1}]
PASS P7 dim rel(Lam) = 1 (it is Z)


   PSLQ on (pi, Lam_1..8): 2 [{0: 1, 1: -4, 4: 2, 5: 1, 6: 1}, {0: 2, 2: -8, 3: -4, 4: -4, 7: 1}]
PASS P7 dim rel(pi, Lam) = 2


PSLQ, run on its own, returns the relation basis $\{Z,\ \pi=4\Lambda_1-2\Lambda_4-\Lambda_5-\Lambda_6\ \text{(BBP)},\ \pi=4\Lambda_2+2\Lambda_3+2\Lambda_4-\tfrac12\Lambda_7\}$. The third relation is the $r=\tfrac12$ machine, which § 5 shows is BBP read backwards.

## 4 · Every π machine silences the two primes

The lane polynomials factor as follows (exact):

$$P_{\rm BBP}=-x(x-1)(x^2+2)(x^2+2x+2),\qquad P_W=-\tfrac14x(x^2-2)(x^2+2)(x^2+2x+2),$$
$$P_Z=-x(x^2+2)(x^2+2x+2)(x^2-4x+2),\qquad P_{c(1/2)}=-\tfrac12x^2(x-2)(x^2+2)(x^2+2x+2).$$

Each contains $(x^2+2)(x^2+2x+2)$, so each vanishes on the prime-3 and prime-5 orbits. A π machine is a polynomial that silences every return it does not want. What separates the machines is how they split between the two prime-free returns: the phase wheel at $1-i$ and the scale wheel at $\sqrt2$.

In [6]:
want = {"BBP": (c_BBP, -x*(x-1)*(x**2+2)*(x**2+2*x+2)),
        "wheel": (c_W, -sp.Rational(1,4)*x*(x**2-2)*(x**2+2)*(x**2+2*x+2)),
        "Z": (Z, -x*(x**2+2)*(x**2+2*x+2)*(x**2-4*x+2)),
        "mirror": (c_of(sp.Rational(1,2)), -sp.Rational(1,2)*x**2*(x-2)*(x**2+2)*(x**2+2*x+2))}
print(f"{'machine':8s} {'factored lane polynomial':55s} P(sqrt2)  P(1-i)        P(sqrt2 i) P(-1-i)")
for k, (cv, target) in want.items():
    P = poly(cv); ev = [sp.expand(P.subs(x, v)) for v in (S2, 1 - I, S2 * I, -1 - I)]
    print(f"{k:8s} {str(sp.factor(P)):55s} {str(ev[0]):8s}  {str(ev[1]):12s}  {str(ev[2]):9s}  {ev[3]}")
    check(f"P2 factorization of P_{k}", sp.expand(P - target) == 0)
    check(f"P4 P_{k} silent on both prime orbits", ev[2] == 0 and ev[3] == 0)
check("P4 orbit values (P(sqrt2),P(1-i)): BBP (-16,16-16i) wheel (0,-16i) Z (64,-64) mirror (16,-16-16i)",
      [tuple(sp.expand(poly(want[k][0]).subs(x, v)) for v in (S2, 1 - I)) for k in want] ==
      [(-16, 16 - 16*I), (0, -16*I), (64, -64), (16, -16 - 16*I)])

machine  factored lane polynomial                                P(sqrt2)  P(1-i)        P(sqrt2 i) P(-1-i)
BBP      -x*(x - 1)*(x**2 + 2)*(x**2 + 2*x + 2)                  -16       16 - 16*I     0          0
PASS P2 factorization of P_BBP
PASS P4 P_BBP silent on both prime orbits
wheel    -x*(x**2 - 2)*(x**2 + 2)*(x**2 + 2*x + 2)/4             0         -16*I         0          0
PASS P2 factorization of P_wheel
PASS P4 P_wheel silent on both prime orbits
Z        -x*(x**2 + 2)*(x**2 - 4*x + 2)*(x**2 + 2*x + 2)         64        -64           0          0
PASS P2 factorization of P_Z
PASS P4 P_Z silent on both prime orbits
mirror   -x**2*(x - 2)*(x**2 + 2)*(x**2 + 2*x + 2)/2             16        -16 - 16*I    0          0
PASS P2 factorization of P_mirror
PASS P4 P_mirror silent on both prime orbits
PASS P4 orbit values (P(sqrt2),P(1-i)): BBP (-16,16-16i) wheel (0,-16i) Z (64,-64) mirror (16,-16-16i)


In this table the wheel is the *pure* point: $P_W(\sqrt2)=0$ and $P_W(1-i)=-16i$. It reads only its own phase return and carries no $\ln2$ at all. BBP borrows $+16$ on the phase return and $-16$ on the scale return, and the mirror borrows the opposite signs. The borrowed $\ln2$ cancels in both. That collision is what buys BBP its sparsity: 4 lanes against the wheel's 6.

## 5 · Reverse it: the mirror

Define $R$ on lane space by lane $j\mapsto 8-j$ with weight $-2^{\,j-4}$, and lane $8\mapsto-$lane 8. On the roots, $P_{R(c)}(\gamma)=-P_c(2/\gamma)=-P_c(\bar\gamma)$. On the wheels, $w\mapsto\bar w=1/(2w)$: the wheel run backwards, rescaled by the one bit.

$R$ is a value-preserving involution, and:

$$R(c_{\rm BBP})=c(\tfrac12),\qquad R(c_W)=c_W,\qquad Z=2\bigl(R(c_{\rm BBP})-c_{\rm BBP}\bigr).$$

The zero route is BBP read backwards minus BBP read forwards. These are two traversals whose artifacts annihilate. The wheel is the fixed point of the reflection, the place where the two readings cancel. *Say it, then reverse it.*

In [7]:
def Rm(v):
    v = [sp.Rational(t) for t in v]; out = [sp.Integer(0)] * 8
    for j in range(1, 8): out[8 - j - 1] += -sp.Integer(2) ** (j - 4) * v[j - 1]
    out[7] += -v[7]; return out
half = c_of(sp.Rational(1, 2))
check("P5 R is an involution", all(Rm(Rm(v)) == [sp.Rational(t) for t in v] for v in (c_BBP, Z, c_W, c_of(sp.Rational(1, 3)))))
check("P5 R(c_BBP) = c(1/2): the mirror is BBP read backwards", Rm(c_BBP) == half)
check("P5 R(c_W) = c_W: the wheel is self-mirror", Rm(c_W) == [sp.Rational(t) for t in c_W])
check("P5 Z = 2(R(c_BBP) - c_BBP)", [2 * (a - sp.Rational(b)) for a, b in zip(Rm(c_BBP), c_BBP)] == [sp.Integer(t) for t in Z])
gs = [S2 * sp.exp(-I * sp.pi * m / 4) for m in range(8)]
check("P5 P_R(c)(g) = -P_c(2/g) on all 8 roots", all(sp.simplify(sp.expand_complex(poly(Rm(c_BBP)).subs(x, g) + poly(c_BBP).subs(x, 2 / g))) == 0 for g in gs))
check("P5 R preserves value: R(BBP).Lam = pi", fabs(dot(Rm(c_BBP), L8) - pi) < TOL)

PASS P5 R is an involution
PASS P5 R(c_BBP) = c(1/2): the mirror is BBP read backwards
PASS P5 R(c_W) = c_W: the wheel is self-mirror
PASS P5 Z = 2(R(c_BBP) - c_BBP)
PASS P5 P_R(c)(g) = -P_c(2/g) on all 8 roots
PASS P5 R preserves value: R(BBP).Lam = pi


## 6 · The interface: deformation is free, computation is not

The whole π-line is one pencil (**unsealed**, found after the run, verified exactly):

$$P_{c(r)}=-x(x^2+2)(x^2+2x+2)\,\bigl[(x-1)+r(x^2-4x+2)\bigr].$$

The three special machines are the three that vanish at the reader points $x=1,\ \sqrt2,\ 2$, a geometric run with ratio $\sqrt2$, one tick of the wheel's scale $|w|=1/\sqrt2$:

* BBP ($r=0$) vanishes at $x=1$;
* the wheel ($r=\tfrac14$) vanishes at $x=\sqrt2$;
* the mirror ($r=\tfrac12$) vanishes at $x=2$.

The reversal $x\mapsto2/x$ swaps $1\leftrightarrow2$ and fixes $\sqrt2$.

$x=1$ is the plain reader. $P_c(1)=\sum_jc_j$ is the machine's weight sum, which is the carry drift of the column machine, and $P_{c(r)}(1)=15r$. Moving along $Z$ changes nothing in value: that is the passive deformation. Its interface cost is $15$ per unit of $r$, the full $2^4-1$. BBP is the unique zero-drift point on the line. The digit machines below run the same column addition as § 119 / § 124 on integer weights $4c(r)$, computing $4\pi$.

In [8]:
P_line = sp.expand(-x*(x**2+2)*(x**2+2*x+2)*((x-1) + r*(x**2-4*x+2)))
check("pencil: P_c(r) = -x(x^2+2)(x^2+2x+2)[(x-1)+r(x^2-4x+2)]",
      sp.expand(P_line - sum((c_BBP[j] + r*Z[j]) * x**(j+1) for j in range(8))) == 0)
print("   weight sum P_c(r)(1) =", sp.factor(P_line.subs(x, 1)))
for rv, xv in [(0, 1), (sp.Rational(1, 4), S2), (sp.Rational(1, 2), 2)]:
    check(f"r = {rv}: P_c(r)({xv}) = 0", sp.simplify(P_line.subs({r: rv, x: xv})) == 0)

def ord16(o):
    if o == 1: return 0
    t, v = 1, 16 % o
    while v != 1: v, t = v * 16 % o, t + 1
    return t
def machine(W, N=64, G=16):
    Cn = N + G; cols = [0] * (Cn + 1); reads = ringlen = 0
    for k in range(Cn):
        for j, wt in W.items():
            d = 8 * k + j
            if d == 1: cols[0] += wt; continue
            o = d
            while o % 2 == 0: o //= 2
            ringlen += ord16(o); rr = 1 % d
            for t in range(1, Cn - k + 1):
                q, rr = divmod(16 * rr, d); cols[k + t] += wt * q; reads += 1
    colmax = max(abs(v) for v in cols[1:]); carry, cs = 0, []
    for i in range(Cn, 0, -1):
        v = cols[i] + carry; cols[i], carry = v % 16, v // 16; cs.append(carry)
    cols[0] += carry
    return format(cols[0], "x") + "." + "".join(format(v, "x") for v in cols[1:N + 1]), reads, ringlen, colmax, (min(cs), max(cs))
def machin4pi(N):
    def at(xx, one):
        s, t, k, sg = 0, one // xx, 1, 1
        while t: s += sg * (t // k); t //= xx * xx; k += 2; sg = -sg
        return s
    one = 16 ** (N + 12); h = format(4 * (16 * at(5, one) - 4 * at(239, one)) // 16 ** 12, "x")
    return h[0] + "." + h[1:N + 1]
ref = machin4pi(64); print("   Machin 4*pi:", ref)
print(f"   {'machine':13s} {'lanes':18s} {'weight sum':>10s} {'ring reads':>10s} {'ring len':>9s} {'max|col|':>9s}  carry range")
for name, W in [("r=0 BBP", {1: 16, 4: -8, 5: -4, 6: -4}), ("r=1/4 wheel", {1: 8, 2: 8, 3: 4, 5: -2, 6: -2, 7: -1}),
                ("r=1/2 mirror", {2: 16, 3: 8, 4: 8, 7: -2})]:
    h, reads, rl, cm, cr = machine(W)
    print(f"   {name:13s} {str(sorted(W)):18s} {sum(W.values()):>10d} {reads:>10d} {rl:>9d} {cm:>9d}  {list(cr)}")
    check(f"P9 {name}: 64 hex digits of 4*pi = Machin", h == ref)

PASS pencil: P_c(r) = -x(x^2+2)(x^2+2x+2)[(x-1)+r(x^2-4x+2)]
   weight sum P_c(r)(1) = 15*r
PASS r = 0: P_c(r)(1) = 0
PASS r = 1/4: P_c(r)(sqrt(2)) = 0
PASS r = 1/2: P_c(r)(2) = 0
   Machin 4*pi: c.90fdaa22168c234c4c6628b80dc1cd129024e088a67cc74020bbea63b139b225
   machine       lanes              weight sum ring reads  ring len  max|col|  carry range
   r=0 BBP       [1, 4, 5, 6]                0      12880      9461      2380  [-89, 149]
PASS P9 r=0 BBP: 64 hex digits of 4*pi = Machin
   r=1/4 wheel   [1, 2, 3, 5, 6, 7]         15      19360     19477      8539  [1, 563]
PASS P9 r=1/4 wheel: 64 hex digits of 4*pi = Machin
   r=1/2 mirror  [2, 3, 4, 7]               30      12960     12226     16404  [5, 1088]
PASS P9 r=1/2 mirror: 64 hex digits of 4*pi = Machin


## 7 · One layer down, base −4: no collision, one machine

$x^4+4=(x^2-2x+2)(x^2+2x+2)$ contains only the phase wheel and the prime-5 return. That gives 4 lanes, 4 constants, and no zero route. The unique π machine has polynomial $x(x^2+2x+2)$, which silences prime 5:

$$\pi=\sum_{k\ge0}\Bigl(-\frac14\Bigr)^k\Bigl[\frac2{4k+1}+\frac2{4k+2}+\frac1{4k+3}\Bigr].$$

This is the $s$-wheel itself; it matches GPT's $A_j$ regrouping. The zero route first appears one layer up, where $x^8-16=(x^4+4)(x^4-4)$ brings in the scale wheel $x^2-2$, whose return shares $\ln2$ with the phase wheel. (§ 125.6's "$\pi/4=\sum 2/((4j+1)(4j+3))$" is true, but it is paired Leibniz, not a regrouping of the wheel. The label is corrected here.)

In [9]:
mp.dps = 420
L4 = lanes(4, -4)
d4, r4 = reldim(L4); print("   PSLQ on L'_1..4:", d4, r4); check("P7 base -4: no relation among the 4 lanes", d4 == 0)
d5, r5 = reldim([pi] + L4); print("   PSLQ on (pi, L'):", d5, r5)
check("P7 base -4: unique pi machine pi = 2L'1 + 2L'2 + L'3", d5 == 1 and r5[0] in ({0: -1, 1: 2, 2: 2, 3: 1}, {0: 1, 1: -2, 2: -2, 3: -1}))
print("   its polynomial:", sp.factor(2*x + 2*x**2 + x**3), "  (x^4+4 =", sp.factor(x**4 + 4), ")")

   PSLQ on L'_1..4: 0 []
PASS P7 base -4: no relation among the 4 lanes
   PSLQ on (pi, L'): 1 [{0: 1, 1: -2, 2: -2, 3: -1}]
PASS P7 base -4: unique pi machine pi = 2L'1 + 2L'2 + L'3
   its polynomial: x*(x**2 + 2*x + 2)   (x^4+4 = (x**2 - 2*x + 2)*(x**2 + 2*x + 2) )


## 8 · Up the register stack

Take base $2^{n/2}$ with $n$ lanes. The wheels are $w_m=\zeta_n^m/\sqrt2$, the returns are $\beta_m=\sqrt2-\zeta_n^m$, and

$$\prod_{m\bmod n}\beta_m=2^{n/2}-1 .$$

Each orbit factor of $x^n-2^{n/2}$, evaluated at the reader point $x=1$, is $\pm$ that orbit's return product. So the plain reader sees the $\tfrac n2$-bit register's full ring.

On the chain $n=2^e$ the orbits are:

* the scale wheel (unit);
* the prime-3 orbit ($\zeta$-order 4);
* the order-8 level, which splits in two: the phase wheel (torsion) and the prime-5 orbit;
* one orbit per Fermat number $F_{e-2}$ at each order $2^e\ge16$.

At $n=128$ this list is exactly § 117's register-stack primes $3,5,17,257,65537,641\cdot6700417$.

Every prime factor at order $2^e\ge16$ is $\equiv1\pmod{2^e}$ (Lucas, STANDS). It therefore splits completely and is owned by one orbit alone. No prime is shared and no new unit appears, so the only collision on the chain is phase against scale.

**Prediction (sealed):** $Z(n)=1$ at every layer of the chain. Off the chain, where $3^2\mid 2^{n/2}-1$ and extra units appear, the count is measured, not predicted.

In [10]:
for n in (4, 8, 12, 16, 24, 32, 64, 128):
    mp.dps = 120
    tot, desc = mpc(1), []
    for o in galois_orbits(n):
        pr = mpc(1)
        for m in o: pr *= sqrt(2) - exp(2j * pi * m / n)
        iv = int(nint(pr.real)); assert fabs(pr.imag) < 1e-80 and fabs(pr.real - iv) < 1e-80
        tors = all(fabs(abs(sqrt(2) - exp(2j * pi * m / n)) - 1) < 1e-80 for m in o)
        desc.append(("torsion" if tors else "unit") if abs(iv) == 1 else str(abs(iv))); tot *= pr
    ok = fabs(tot - (2 ** (n // 2) - 1)) < 1e-60
    print(f"   n={n:3d}  2^{n//2}-1 = {sp.factorint(2**(n//2)-1)}  orbit returns: {', '.join(desc)}")
    check(f"P8 n={n}: prod of returns = 2^(n/2)-1", ok)
for k in range(2, 6):
    for p in sp.factorint(2 ** (2 ** k) + 1):
        check(f"P8 F_{k} prime {p} = 1 mod 2^{k+2} (splits completely at its layer)", (p - 1) % 2 ** (k + 2) == 0)
for n in (8, 16, 32):
    vals = sorted(abs(int(g.subs(x, 1))) for g, _ in sp.factor_list(x ** n - 2 ** (n // 2))[1])
    check(f"reader point x=1 reads the orbits of x^{n}-2^{n//2} as {vals}", sp.prod(vals) == 2 ** (n // 2) - 1)

   n=  4  2^2-1 = {3: 1}  orbit returns: unit, 3
PASS P8 n=4: prod of returns = 2^(n/2)-1
   n=  8  2^4-1 = {3: 1, 5: 1}  orbit returns: unit, torsion, 3, 5
PASS P8 n=8: prod of returns = 2^(n/2)-1
   n= 12  2^6-1 = {3: 2, 7: 1}  orbit returns: unit, 3, 7, 3
PASS P8 n=12: prod of returns = 2^(n/2)-1
   n= 16  2^8-1 = {3: 1, 5: 1, 17: 1}  orbit returns: unit, 17, torsion, 3, 5
PASS P8 n=16: prod of returns = 2^(n/2)-1
   n= 24  2^12-1 = {3: 2, 5: 1, 7: 1, 13: 1}  orbit returns: unit, unit, 3, torsion, 7, 13, 3, 5
PASS P8 n=24: prod of returns = 2^(n/2)-1
   n= 32  2^16-1 = {3: 1, 5: 1, 17: 1, 257: 1}  orbit returns: unit, 257, 17, torsion, 3, 5
PASS P8 n=32: prod of returns = 2^(n/2)-1
   n= 64  2^32-1 = {3: 1, 5: 1, 17: 1, 257: 1, 65537: 1}  orbit returns: unit, 65537, 257, 17, torsion, 3, 5
PASS P8 n=64: prod of returns = 2^(n/2)-1
   n=128  2^64-1 = {3: 1, 5: 1, 17: 1, 257: 1, 641: 1, 65537: 1, 6700417: 1}  orbit returns: unit, 4294967297, 65537, 257, 17, torsion, 3, 5
PASS P8 n=128:

PASS reader point x=1 reads the orbits of x^32-2^16 as [1, 1, 3, 5, 17, 257]


In [11]:
# zero-route dimension Z(n) by PSLQ (|coeff| <= 1e12); pi, pi*sqrt3, pi*sqrt2 membership in the lane span
ZN = {}
for n, dps in [(8, 300), (12, 400), (16, 500)] + ([(24, 700), (32, 800)] if RUN_SLOW else []):
    mp.dps = dps; Ln = lanes(n, 2 ** (n // 2), dps); t0 = time.time()
    rels, basis = span_basis(Ln); zn = len(rels); ZN[n] = zn
    mem = {nm: in_span(cv, Ln, basis) for nm, cv in [("pi", pi), ("pi*sqrt3", pi * sqrt(3)), ("pi*sqrt2", pi * sqrt(2))]}
    print(f"   n={n:2d} (base 2^{n//2}): Z(n) = {zn}; in lane span: {mem}   [{time.time()-t0:.0f}s]")
    for rl in rels: print("        relation:", rl)
for n in (8, 16, 32):
    if n in ZN: check(f"P8 on-chain Z({n}) = 1", ZN[n] == 1)
for n in (12, 24):
    if n in ZN: check(f"P8 off-chain Z({n}) >= 1", ZN[n] >= 1)

   n= 8 (base 2^4): Z(n) = 1; in lane span: {'pi': True, 'pi*sqrt3': False, 'pi*sqrt2': False}   [1s]
        relation: {0: -8, 1: 8, 2: 4, 3: 8, 4: 2, 5: 2, 6: -1}


   n=12 (base 2^6): Z(n) = 1; in lane span: {'pi': False, 'pi*sqrt3': True, 'pi*sqrt2': True}   [8s]
        relation: {1: -16, 3: 24, 5: 8, 7: 6, 9: -1}


   n=16 (base 2^8): Z(n) = 1; in lane span: {'pi': True, 'pi*sqrt3': False, 'pi*sqrt2': False}   [61s]
        relation: {0: 128, 1: -128, 2: -64, 3: -128, 4: -32, 5: -32, 6: 16, 8: 8, 9: -8, 10: -4, 11: -8, 12: -2, 13: -2, 14: 1}


   n=24 (base 2^12): Z(n) = 5; in lane span: {'pi': True, 'pi*sqrt3': True, 'pi*sqrt2': True}   [247s]
        relation: {4: 256, 5: -256, 6: -128, 8: -128, 9: 128, 10: 64, 11: 64, 13: 16, 15: -24, 16: -4, 17: 4, 18: 2, 19: 2, 20: 2, 21: 3, 22: -1}
        relation: {0: 256, 1: 256, 2: -256, 3: -512, 4: -192, 5: -448, 6: -64, 7: -288, 8: -64, 9: -48, 11: -16, 12: -4, 13: 16, 14: 4, 15: -6, 16: 3, 17: -1, 18: 1, 19: -3, 20: 1}
        relation: {0: -256, 1: 768, 2: -256, 4: -64, 5: -64, 6: -64, 7: -96, 9: -16, 11: 16, 12: 4, 13: 16, 14: 4, 15: 6, 16: 1, 17: 1, 18: 1, 19: -1}
        relation: {0: 1024, 1: -1536, 2: -512, 3: -256, 4: -128, 5: -128, 6: 64, 7: 192, 9: -32, 11: -32, 12: -16, 13: -16, 14: 8, 16: 2, 17: 2, 18: -1}
        relation: {0: -4608, 1: 5632, 2: 4608, 4: 1152, 5: 640, 7: -960, 9: 352, 11: 96, 12: 72, 13: 16, 14: -72, 15: -60, 16: -18, 17: -26, 21: 1}


   n=32 (base 2^16): Z(n) = 1; in lane span: {'pi': True, 'pi*sqrt3': False, 'pi*sqrt2': False}   [2888s]
        relation: {0: 32768, 1: -32768, 2: -16384, 3: -32768, 4: -8192, 5: -8192, 6: 4096, 8: 2048, 9: -2048, 10: -1024, 11: -2048, 12: -512, 13: -512, 14: 256, 16: 128, 17: -128, 18: -64, 19: -128, 20: -32, 21: -32, 22: 16, 24: 8, 25: -8, 26: -4, 27: -8, 28: -2, 29: -2, 30: 1}
PASS P8 on-chain Z(8) = 1
PASS P8 on-chain Z(16) = 1
PASS P8 on-chain Z(32) = 1
PASS P8 off-chain Z(12) >= 1
PASS P8 off-chain Z(24) >= 1


## 9 · Two sheets (§ 127's open hinge)

§ 127 asked whether a LEN-9 ring forces the twisted two-sheet route (18 positions, step $\pi/9$). It does not force it, because neither direction is forced. Both are native shift-register transitions:

* the plain ring (rotate), with period $n$;
* the twisted ring (rotate and complement the wrapped bit; the Johnson / Möbius counter), with period $2n$.

After $n$ twisted steps every word lands on its complement. The edge-difference reader $D$ sees period $n$ along the twisted route, so it cannot tell which direction it is on. This is **unsealed**.

In [12]:
def Dr(wd, n): return wd ^ (((wd << 1) | (wd >> (n - 1))) & ((1 << n) - 1))
def plain(wd, n): return ((wd << 1) | (wd >> (n - 1))) & ((1 << n) - 1)
def twist(wd, n): return ((wd << 1) | (1 ^ (wd >> (n - 1)))) & ((1 << n) - 1)
def per(f, n, w0=1):
    v, t = f(w0, n), 1
    while v != w0: v, t = f(v, n), t + 1
    return t
for n in (3, 5, 7, 9, 11):
    seq, v = [], 1
    for _ in range(2 * n): seq.append(Dr(v, n)); v = twist(v, n)
    dper = next(p for p in range(1, 2 * n + 1) if all(seq[i] == seq[(i + p) % (2 * n)] for i in range(2 * n)))
    def tn(wd):
        for _ in range(n): wd = twist(wd, n)
        return wd
    comp = all(tn(wd) == wd ^ ((1 << n) - 1) for wd in range(1 << n))
    print(f"   LEN {n:2d}: plain period {per(plain, n):2d}  twisted period {per(twist, n):2d}  D-period on twisted route {dper:2d}  twist^n = complement: {comp}")
    check(f"two sheets at LEN {n}", per(plain, n) == n and per(twist, n) == 2 * n and dper == n and comp)

   LEN  3: plain period  3  twisted period  6  D-period on twisted route  3  twist^n = complement: True
PASS two sheets at LEN 3
   LEN  5: plain period  5  twisted period 10  D-period on twisted route  5  twist^n = complement: True
PASS two sheets at LEN 5
   LEN  7: plain period  7  twisted period 14  D-period on twisted route  7  twist^n = complement: True
PASS two sheets at LEN 7
   LEN  9: plain period  9  twisted period 18  D-period on twisted route  9  twist^n = complement: True
PASS two sheets at LEN 9
   LEN 11: plain period 11  twisted period 22  D-period on twisted route 11  twist^n = complement: True
PASS two sheets at LEN 11


## 9b · § 128's lifted p-wheel: when is its degree floor attained?

$w_p=\zeta_{4p}/\sqrt p$ satisfies $|w|^2=1/p$, $w^{2p}=-1/p^p$ and $w^{4p}=1/p^{2p}$. § 128 bounded $[\mathbb Q(w_p):\mathbb Q]\ge\varphi(2p)$ and left open whether the bound is attained.

Sealed separately (`0091f18c…`, text in the cell below): $w_p^2=\zeta_{2p}/p$ generates $\mathbb Q(\zeta_p)$, so $w_p\in\mathbb Q(\zeta_p)$ iff $-p$ is a square there. By the quadratic Gauss sum (STANDS) that holds iff $p\equiv3\pmod4$. So the degree is $p-1$ for $p\equiv3$, and $2(p-1)$ for $p\equiv1$, where the wheel must also carry $i$.

In [13]:
LIFT_SEAL = 'NEXUS section 128 follow-up - sealed before run, 17 Oct 2026.\nw_p = zeta_(4p)/sqrt(p). Exact: |w|^2 = 1/p, w^(2p) = -1/p^p, w^(4p) = 1/p^(2p).\nDegree law: [Q(w_p):Q] = 2 at p=2; = p-1 if p = 3 mod 4 (lower bound phi(2p) attained, w_p in Q(zeta_p));\n= 2(p-1) if p = 1 mod 4 (bound not attained, w_p needs i: Q(w_p) = Q(zeta_(4p))).\nReason: w_p^2 = zeta_(2p)/p generates Q(zeta_p); w_p in Q(zeta_p) iff -p is a square in Q(zeta_p)\niff p = 3 mod 4 (quadratic Gauss sum: sqrt((-1)^((p-1)/2) p) in Q(zeta_p)).\nCheck p = 2, 3, 5, 7, 11, 13 by exact minimal polynomials.\n'
print("lift seal SHA-256:", hashlib.sha256(LIFT_SEAL.encode()).hexdigest())
check("lift seal matches 0091f18c...", hashlib.sha256(LIFT_SEAL.encode()).hexdigest() == "0091f18c6e1559afe046ee6982ef354e42ffa0bfdb9bf79ea0c9b86411329302")
for p in [2, 3, 5, 7, 11, 13]:
    wp = sp.exp(I * sp.pi / (2 * p)) / sp.sqrt(p)
    ids = (sp.simplify(sp.expand_complex(wp * sp.conjugate(wp)) - sp.Rational(1, p)) == 0,
           sp.simplify(sp.expand_complex(wp ** (2 * p)) + sp.Rational(1, p ** p)) == 0,
           sp.simplify(sp.expand_complex(wp ** (4 * p)) - sp.Rational(1, p ** (2 * p))) == 0)
    d = int(sp.degree(sp.minimal_polynomial(wp, x), x))
    want = 2 if p == 2 else (p - 1 if p % 4 == 3 else 2 * (p - 1))
    print(f"   p={p:2d}  p mod 4 = {p % 4}  [Q(w_p):Q] = {d:2d}  phi(2p) = {int(sp.totient(2 * p))}")
    check(f"p={p}: identities exact; degree {want} as sealed", all(ids) and d == want)

lift seal SHA-256: 0091f18c6e1559afe046ee6982ef354e42ffa0bfdb9bf79ea0c9b86411329302
PASS lift seal matches 0091f18c...
   p= 2  p mod 4 = 2  [Q(w_p):Q] =  2  phi(2p) = 2
PASS p=2: identities exact; degree 2 as sealed
   p= 3  p mod 4 = 3  [Q(w_p):Q] =  2  phi(2p) = 2
PASS p=3: identities exact; degree 2 as sealed
   p= 5  p mod 4 = 1  [Q(w_p):Q] =  8  phi(2p) = 4
PASS p=5: identities exact; degree 8 as sealed
   p= 7  p mod 4 = 3  [Q(w_p):Q] =  6  phi(2p) = 6
PASS p=7: identities exact; degree 6 as sealed


   p=11  p mod 4 = 3  [Q(w_p):Q] = 10  phi(2p) = 10


PASS p=11: identities exact; degree 10 as sealed


   p=13  p mod 4 = 1  [Q(w_p):Q] = 24  phi(2p) = 12
PASS p=13: identities exact; degree 24 as sealed


## 10 · Ledger

| claim | status |
|---|---|
| $c_W=c_{\rm BBP}+\tfrac14Z$; the wheel's base-16 regrouping is BBP plus a quarter of BBP's zero relation | RUNS (exact); closes GPT's open "further algebraic identity" |
| $Z\cdot\Lambda=0$; the π machines form the line $c_{\rm BBP}+rZ$ | RUNS (420 digits, exact kernel); zero relation STANDS (BBP 1997) |
| a machine is read through $P_c$ at the four orbits of $x^8-16$ | RUNS (exact + numeric) |
| returns: scale (unit), phase (torsion), prime 3, prime 5; $\prod\beta=15$ | RUNS (exact norms) |
| 8 lanes, 7 constants, one zero route = the phase/scale collision on $\ln2$ | RUNS (rank, kernel, PSLQ); independence STANDS (Baker) |
| every π machine silences both primes; factorizations of $P_{\rm BBP},P_W,P_Z,P_{c(1/2)}$ | RUNS (exact) |
| mirror $R$ ($w\mapsto\bar w$): $R(\rm BBP)=c(\tfrac12)$, $R(c_W)=c_W$, $Z=2(R(\rm BBP)-\rm BBP)$ | RUNS (exact) |
| pencil; BBP / wheel / mirror vanish at $x=1,\sqrt2,2$; weight sum $15r$ | RUNS (exact), **unsealed** |
| three digit machines: identical 64 hex digits of $4\pi$; interface costs differ | RUNS |
| base −4: no zero route; unique π machine $=$ the wheel | RUNS (PSLQ) |
| $\prod_m(\sqrt2-\zeta_n^m)=2^{n/2}-1$; on the chain the orbit returns are § 117's primes | RUNS ($n\le128$); identity STANDS |
| $Z(n)=1$ on the register chain | RUNS at $n=8,16$ (+32 if `RUN_SLOW`); proof for all layers OPEN (needs: no unit beyond scale/phase on the chain, no shared prime) |
| off-chain $Z(12)=1$, $\pi\notin$ span, $\pi\sqrt3,\pi\sqrt2\in$ span | RUNS (PSLQ), **unsealed** |
| two sheets: plain ring period $n$, twisted ring $2n$, reader $D$ blind to the twist | RUNS, **unsealed** |
| $[\mathbb Q(w_p):\mathbb Q]=p-1$ ($p\equiv3$), $2(p-1)$ ($p\equiv1$), 2 ($p=2$) | RUNS $p\le13$ (sealed `0091f18c`); law STANDS (Gauss sum) |
| § 125.6 label: $\sum2/((4j+1)(4j+3))=\pi/4$ is paired Leibniz, not a wheel regrouping | corrected |

In [14]:
SEAL = "NEXUS zero-route decompile - sealed predictions, 17 Oct 2026, before any run.\nLanes Lam_j = sum_k 16^-k/(8k+j), j=1..8 (base 16). Lane polynomial P_c(x) = sum_j c_j x^j.\nP1  c_W=(2,2,1,0,-1/2,-1/2,-1/4,0) computes pi; c_W - c_BBP = Z/4, Z=(-8,8,4,8,2,2,-1,0); Z.Lam = 0.\nP2  P_BBP=-x(x-1)(x^2+2)(x^2+2x+2); P_W=-(1/4)x(x^2-2)(x^2+2)(x^2+2x+2);\n    P_Z=-x(x^2+2)(x^2+2x+2)(x^2-4x+2); P_c(1/2)=-(1/2)x^2(x-2)(x^2+2)(x^2+2x+2).\nP3  x^8-16=(x^2-2)(x^2+2)(x^2-2x+2)(x^2+2x+2). Returns beta_m = sqrt2 - zeta8^m:\n    m in {1,7} roots of unity; {0,4} units; {2,6} norm 9; {3,5} norm 25 (norms in Q(zeta8)); prod_m beta_m = 15.\nP4  (P(sqrt2), P(1-i)): BBP (-16, 16-16i); W (0,-16i); Z (64,-64); c(1/2) (16,-16-16i);\n    all four vanish at sqrt2*i and -1-i.\nP5  mirror R: lane j -> 8-j with weight -2^(j-4) (w -> conj w = 1/(2w)) is an involution, value-preserving;\n    R(c_BBP)=c(1/2); R(c_W)=c_W; Z = 2(R(c_BBP)-c_BBP).\nP6  support of c(r)=c_BBP+rZ: 4 at r in {0,1/2}; 6 at r=1/4; 7 at other r.\nP7  PSLQ: relation space of (Lam_1..Lam_8) has dim 1; of (pi,Lam_1..Lam_8) dim 2.\n    Base -4 lanes L'_j=sum_k(-1/4)^k/(4k+j), j=1..4: no relation among L'; (pi,L') dim 1, pi=2L'1+2L'2+L'3.\n    The 7 constants {ln2, pi, sqrt2 ln(1+sqrt2), ln3, sqrt2 atan(1/sqrt2), ln5, atan(1/3)}: no integer\n    relation with |coeff|<=1e12 at 300 digits; each Lam_j is an exact rational combination of them.\nP8  prod_{m mod n}(sqrt2 - zeta_n^m) = 2^(n/2)-1 for even n. On the register chain n=2^e: orbits are\n    {zeta order<=2} unit, {order 4} prime 3, {order 8} split into torsion (product 1) and prime 5,\n    {order 2^e>=16} one orbit with product F_(e-2)=2^(2^(e-2))+1, each prime factor = 1 mod 2^e.\n    Zero-route dimension Z(n)=1 at n=8,16,32. Off-chain n=12,24: Z(n)>=1, measured, not predicted.\nP9  digit machines with integer weights 4c(r), r=0,1/4,1/2: identical 64 hex digits (of 4*pi), equal to\n    Machin; lane sets {1,4,5,6}, {1,2,3,5,6,7}, {2,3,4,7}.\n"
h = hashlib.sha256(SEAL.encode()).hexdigest()
print("seal SHA-256:", h); check("seal matches e80c1cc4...", h == "e80c1cc4b724db85a759c3568ebb0ae5e39a66352e94b088040900c9d03bd63d")
print(f"\n{sum(checks.values())}/{len(checks)} checks pass;", "ALL PASS" if all(checks.values()) else "FAILURES: " + str([k for k, v in checks.items() if not v]))

seal SHA-256: e80c1cc4b724db85a759c3568ebb0ae5e39a66352e94b088040900c9d03bd63d
PASS seal matches e80c1cc4...

75/75 checks pass; ALL PASS
